# Notebook 04: Deep Learning Models and Expanded Metrics

This notebook trains baseline and FFT-enhanced models, then saves model results, predictions, and classification reports. It now includes metrics needed for confusion matrices, ROC-AUC, PR-AUC, false positive rate, and false negative rate.

## Short Problem Statement

Existing deep learning-based **Intrusion Detection Systems (IDS)** often rely mainly on time-domain flow features and may miss hidden burst patterns, repeated traffic rhythms, and frequency-based attack signatures. **SentinelFlow** addresses this by using **Fast Fourier Transform (FFT)**-enhanced traffic profiling to improve deep learning-based intrusion detection on large-scale network traffic datasets.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'sentinelflow_utils.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from sentinelflow_utils import *
ensure_dirs(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
baseline_path = PROJECT_ROOT / 'data/processed/03_baseline_segments.parquet'
fft_path = PROJECT_ROOT / 'data/processed/03_fft_segments.parquet'
if not baseline_path.exists() or not fft_path.exists():
    raise FileNotFoundError('Run Notebook 03 first to create segment datasets.')
baseline_df = pd.read_parquet(baseline_path)
fft_df = pd.read_parquet(fft_path)
print('Baseline shape:', baseline_df.shape)
print('FFT shape:', fft_df.shape)
display(class_distribution(baseline_df, 'target_attack').head(20))

In [ ]:
# Configuration
RUN_SKLEARN_MODELS = True
RUN_TORCH_DEEP_MODELS = True
RUN_MULTICLASS_MODELS = True
TORCH_EPOCHS = 8
MAX_FEATURES = 250

all_results = []
all_predictions = []
all_reports = {}

if RUN_SKLEARN_MODELS:
    print('Training sklearn binary models...')
    r, p, rep = train_sklearn_binary_models_with_predictions(baseline_df, feature_set='Time-domain baseline', max_features=MAX_FEATURES)
    all_results.append(r); all_predictions.append(p); all_reports.update(rep)
    r, p, rep = train_sklearn_binary_models_with_predictions(fft_df, feature_set='Time-domain + FFT', max_features=MAX_FEATURES)
    all_results.append(r); all_predictions.append(p); all_reports.update(rep)

if RUN_TORCH_DEEP_MODELS:
    print('Training PyTorch binary deep learning models...')
    r, p, rep = train_torch_binary_models_with_predictions(baseline_df, feature_set='Time-domain baseline', epochs=TORCH_EPOCHS, max_features=MAX_FEATURES)
    all_results.append(r); all_predictions.append(p); all_reports.update(rep)
    r, p, rep = train_torch_binary_models_with_predictions(fft_df, feature_set='Time-domain + FFT', epochs=TORCH_EPOCHS, max_features=MAX_FEATURES)
    all_results.append(r); all_predictions.append(p); all_reports.update(rep)

if RUN_MULTICLASS_MODELS:
    print('Training optional multiclass attack-type models...')
    r, p, rep = train_multiclass_models_with_predictions(baseline_df, feature_set='Time-domain baseline', max_features=MAX_FEATURES)
    if len(r): all_results.append(r)
    if len(p): all_predictions.append(p)
    all_reports.update(rep)
    r, p, rep = train_multiclass_models_with_predictions(fft_df, feature_set='Time-domain + FFT', max_features=MAX_FEATURES)
    if len(r): all_results.append(r)
    if len(p): all_predictions.append(p)
    all_reports.update(rep)

results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
predictions_df = pd.concat([p for p in all_predictions if len(p)], ignore_index=True) if all_predictions else pd.DataFrame()

display(results_df.sort_values(['task','macro_f1'], ascending=[True, False]))
print('Predictions shape:', predictions_df.shape)

In [ ]:
metrics_dir = PROJECT_ROOT / 'outputs/metrics'
pred_dir = PROJECT_ROOT / 'outputs/predictions'
report_dir = PROJECT_ROOT / 'reports/classification_reports'
metrics_dir.mkdir(parents=True, exist_ok=True)
pred_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

results_path = metrics_dir / '04_model_results_expanded.csv'
pred_path = pred_dir / '04_all_model_predictions.csv'
legacy_results_path = PROJECT_ROOT / 'outputs/03_model_results.csv'
results_df.to_csv(results_path, index=False)
results_df.to_csv(legacy_results_path, index=False)
if len(predictions_df):
    predictions_df.to_csv(pred_path, index=False)
for name, rep_df in all_reports.items():
    safe = name.replace(' ', '_').replace('/', '_').replace('+', 'plus').replace(':','')
    rep_df.to_csv(report_dir / f'{safe}_classification_report.csv', index=False)
print('Saved:', results_path)
print('Saved:', legacy_results_path)
print('Saved:', pred_path)
print('Saved classification reports to:', report_dir)

In [ ]:
# Quick interpretation helper
if not results_df.empty:
    best = results_df.dropna(subset=['macro_f1']).sort_values('macro_f1', ascending=False).head(5)
    print('Top models by Macro F1-score:')
    display(best[['feature_set','task','model','accuracy','recall','specificity','macro_f1','false_positive_rate','false_negative_rate','roc_auc','pr_auc']])